In [1]:
%%time
%%capture

# install pytabkit, tabicl, and sentence transformer
!pip install pytabkit 
!pip install --quiet tabicl
!pip install -q sentence-transformers

CPU times: user 40.7 ms, sys: 21.6 ms, total: 62.3 ms
Wall time: 14.8 s


In [2]:
%%time

import joblib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import polars as pl

from sklearn.ensemble import VotingRegressor

from sklearn.linear_model import LinearRegression, ElasticNet
from pytabkit import RealMLP_TD_Regressor
from pytabkit import TabM_D_Regressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import random
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix, log_loss,
    mean_absolute_error,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.optimize import minimize, differential_evolution
import lightgbm as lgb
from catboost import CatBoostClassifier
import pickle
import gc
import subprocess, zipfile, os
import xgboost as xgb
import os
from pytabkit import TabM_D_Classifier, TabM_D_Regressor
from pytabkit import RealMLP_TD_Classifier, RealMLP_TD_Regressor
from tabicl import TabICLClassifier, TabICLRegressor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os, gc, re
from collections import Counter
import random
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix, log_loss
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from scipy import stats
from scipy.optimize import minimize, differential_evolution
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import pickle
import gc
import warnings
warnings.filterwarnings("ignore")

model_dir = '/kaggle/working/models/'
data_dir = "/kaggle/working/data/"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

CPU times: user 18.2 s, sys: 3.68 s, total: 21.8 s
Wall time: 52.2 s


In [3]:
# Parse chief complaint structure
def parse_chief_complaint(text):
    """Extract base condition, severity modifier, and trajectory from CC text."""
    if pd.isna(text):
        return "unknown", 0, "none"

    parts = [p.strip() for p in re.split(r"[,\uff0c]", text)]
    base_condition = parts[0]

    severity = 0
    sev_high = ["severe", "massive"] 
    sev_mod  = ["moderate"]
    sev_low  = ["mild", "minor", "light"]

    all_text = text.lower()
    for w in sev_high:
        if w in all_text:
            severity = 3; break
    if severity == 0:
        for w in sev_mod:
            if w in all_text:
                severity = 2; break
    if severity == 0:
        for w in sev_low:
            if w in all_text:
                severity = 1; break

    trajectory = "none"
    traj_map = {"worsening": "worsening", "improving": "improving",
                "intermittent": "intermittent", "constant": "constant",
                "acute": "acute", "chronic": "chronic"}
    for kw, val in traj_map.items():
        if kw in all_text:
            trajectory = val; break

    return base_condition, severity, trajectory



In [4]:
%%time

SEED = 42
N_FOLDS = 5
TARGET = 'ed_los_hours'
ID = 'patient_id'
np.random.seed(SEED)

import subprocess, zipfile
import os

KAGGLE_PATH = '/kaggle/input/competitions/triagegeist'
LOCAL_PATH  = '../data'

if os.path.exists(KAGGLE_PATH) and os.path.isfile(f'{KAGGLE_PATH}/train.csv'):
    DATA = KAGGLE_PATH
elif os.path.exists(LOCAL_PATH) and os.path.isfile(f'{LOCAL_PATH}/train.csv'):
    DATA = LOCAL_PATH
else:
    # Fallback: download competition data via Kaggle API (works on Kaggle runtime)
    DL_DIR = '/kaggle/working/data'
    os.makedirs(DL_DIR, exist_ok=True)
    subprocess.run(['kaggle', 'competitions', 'download', '-c', 'triagegeist', '-p', DL_DIR], check=True)
    for zf in [f for f in os.listdir(DL_DIR) if f.endswith('.zip')]:
        with zipfile.ZipFile(f'{DL_DIR}/{zf}', 'r') as z:
            z.extractall(DL_DIR)
    DATA = DL_DIR
    print(f'Data downloaded to {DL_DIR}')

print(f'Data directory: {DATA}')

train_raw = pd.read_csv(f'{DATA}/train.csv')
test_raw  = pd.read_csv(f'{DATA}/test.csv')
complaints = pd.read_csv(f'{DATA}/chief_complaints.csv')
history    = pd.read_csv(f'{DATA}/patient_history.csv')
sample_sub = pd.read_csv(f'{DATA}/sample_submission.csv')

cc_dedup = complaints[[ID, 'chief_complaint_raw']].drop_duplicates(subset=ID, keep='first')
train = train_raw.merge(cc_dedup, on=ID, how='left').merge(history, on=ID, how='left')
test  = test_raw.merge(cc_dedup, on=ID, how='left').merge(history, on=ID, how='left')

print(f'Train: {train.shape[0]:,} patients x {train.shape[1]} columns')
print(f'Test:  {test.shape[0]:,} patients x {test.shape[1]} columns')
print(f'Chief complaints: {len(complaints):,} records')
print(f'Patient history:  {len(history):,} records ({history.shape[1]-1} comorbidity flags)')

del train_raw, test_raw, complaints, history, sample_sub, cc_dedup
gc.collect()

print()

Data directory: /kaggle/input/competitions/triagegeist
Train: 80,000 patients x 66 columns
Test:  20,000 patients x 63 columns
Chief complaints: 100,000 records
Patient history:  100,000 records (25 comorbidity flags)

CPU times: user 1.3 s, sys: 164 ms, total: 1.47 s
Wall time: 1.79 s


In [5]:
%%time
from sklearn.model_selection import StratifiedKFold, KFold
# All features is made through this function
def features_function(max_words=100, max_chars=90, lim_words=3, lim_chars=3):
    train_raw = pd.read_csv(f'{DATA}/train.csv')
    test_raw  = pd.read_csv(f'{DATA}/test.csv')
    complaints = pd.read_csv(f'{DATA}/chief_complaints.csv')
    history    = pd.read_csv(f'{DATA}/patient_history.csv')
    sample_sub = pd.read_csv(f'{DATA}/sample_submission.csv')
    
    cc_dedup = complaints[[ID, 'chief_complaint_raw']].drop_duplicates(subset=ID, keep='first')
    
    train = train_raw.merge(cc_dedup, on=ID, how='left').merge(history, on=ID, how='left')
    test  = test_raw.merge(cc_dedup, on=ID, how='left').merge(history, on=ID, how='left')

    # Apply parse_chief_complaint to both sets
    parsed = train["chief_complaint_raw"].apply(parse_chief_complaint)
    train["clean_condition"] = [p[0] for p in parsed]
    train["complaint_severity"] = [p[1] for p in parsed]
    
    parsed_test = test["chief_complaint_raw"].apply(parse_chief_complaint)
    test["clean_condition"] = [p[0] for p in parsed_test]
    test["complaint_severity"] = [p[1] for p in parsed_test]

    del parsed, parsed_test, train_raw, test_raw, complaints, history, sample_sub, cc_dedup
    gc.collect()
    
    def engineer_features(df):
        df = df.copy()
        
        # --- VITAL SIGN ABNORMALITY FLAGS ---
        df['flag_hypotension']        = (df['systolic_bp'] < 90).astype(np.int8)
        df['flag_hypertension_crisis'] = (df['systolic_bp'] > 180).astype(np.int8)
        df['flag_severe_hypotension'] = (df['systolic_bp'] < 70).astype(np.int8)
        df['flag_tachycardia']        = (df['heart_rate'] > 100).astype(np.int8)
        df['flag_severe_tachycardia'] = (df['heart_rate'] > 130).astype(np.int8)
        df['flag_bradycardia']        = (df['heart_rate'] < 60).astype(np.int8)
        df['flag_tachypnea']          = (df['respiratory_rate'] > 20).astype(np.int8)
        df['flag_severe_tachypnea']   = (df['respiratory_rate'] > 30).astype(np.int8)
        df['flag_hypothermia']        = (df['temperature_c'] < 36.0).astype(np.int8)
        df['flag_fever']              = (df['temperature_c'] > 38.0).astype(np.int8)
        df['flag_high_fever']         = (df['temperature_c'] > 39.0).astype(np.int8)
        df['flag_hypoxia']            = (df['spo2'] < 92).astype(np.int8)
        df['flag_severe_hypoxia']     = (df['spo2'] < 88).astype(np.int8)
        df['flag_altered_mental']     = (df['gcs_total'] < 15).astype(np.int8)
        df['flag_severe_gcs']         = (df['gcs_total'] <= 8).astype(np.int8)
        df['flag_severe_pain']        = (df['pain_score'] >= 8).astype(np.int8)
        df['flag_high_shock_idx']     = (df['shock_index'] > 1.0).astype(np.int8)
        df['flag_critical_shock_idx'] = (df['shock_index'] > 1.3).astype(np.int8)
        
        flag_cols = [c for c in df.columns if c.startswith('flag_')]
        df['num_abnormal_vitals'] = df[flag_cols].sum(axis=1)
        
        # --- qSOFA SCORE (0-3) ---
        df['qsofa_sbp']  = (df['systolic_bp'] <= 100).astype(np.int8)
        df['qsofa_rr']   = (df['respiratory_rate'] >= 22).astype(np.int8)
        df['qsofa_gcs']  = (df['gcs_total'] < 15).astype(np.int8)
        df['qsofa_score'] = df['qsofa_sbp'] + df['qsofa_rr'] + df['qsofa_gcs']
        
        # --- SIRS-like criteria count ---
        df['sirs_temp']  = ((df['temperature_c'] > 38.0) | (df['temperature_c'] < 36.0)).astype(np.int8)
        df['sirs_hr']    = (df['heart_rate'] > 90).astype(np.int8)
        df['sirs_rr']    = (df['respiratory_rate'] > 20).astype(np.int8)
        df['sirs_count'] = df['sirs_temp'] + df['sirs_hr'] + df['sirs_rr']
        
        # --- COMPOSITE RISK SCORES ---
        hx_cols = [c for c in df.columns if c.startswith('hx_')]
        if hx_cols:
            df['comorbidity_burden'] = df[hx_cols].sum(axis=1)
        
        cv_cols = ['hx_hypertension', 'hx_heart_failure', 'hx_atrial_fibrillation',
                   'hx_coronary_artery_disease', 'hx_stroke_prior', 'hx_peripheral_vascular_disease']
        cv_present = [c for c in cv_cols if c in df.columns]
        df['cv_risk_score'] = df[cv_present].sum(axis=1) if cv_present else 0
        
        resp_cols = ['hx_asthma', 'hx_copd']
        resp_present = [c for c in resp_cols if c in df.columns]
        df['respiratory_risk'] = df[resp_present].sum(axis=1) if resp_present else 0
        
        meta_cols = ['hx_diabetes_type2', 'hx_diabetes_type1', 'hx_obesity',
                     'hx_hypothyroidism', 'hx_hyperthyroidism']
        meta_present = [c for c in meta_cols if c in df.columns]
        df['metabolic_risk'] = df[meta_present].sum(axis=1) if meta_present else 0
        
        psych_cols = ['hx_depression', 'hx_anxiety', 'hx_substance_use_disorder']
        psych_present = [c for c in psych_cols if c in df.columns]
        df['psych_risk'] = df[psych_present].sum(axis=1) if psych_present else 0
        
        # --- AGE-VITAL INTERACTIONS ---
        df['age_x_shock_index']  = df['age'] * df['shock_index']
        df['age_x_news2']        = df['age'] * df['news2_score']
        df['age_x_gcs']          = df['age'] * df['gcs_total']
        df['age_x_comorbidities'] = df['age'] * df['num_comorbidities']
        df['age_x_qsofa']        = df['age'] * df['qsofa_score']
        
        # --- VITAL SIGN RATIOS & PRODUCTS ---
        df['hr_rr_ratio']     = df['heart_rate'] / df['respiratory_rate'].replace(0, np.nan)
        df['sbp_hr_product']  = df['systolic_bp'] * df['heart_rate']
        df['map_hr_ratio']    = df['mean_arterial_pressure'] / df['heart_rate'].replace(0, np.nan)
        df['pp_sbp_ratio']    = df['pulse_pressure'] / df['systolic_bp'].replace(0, np.nan)
        df['spo2_rr_product'] = df['spo2'] * df['respiratory_rate']
        
        # --- ED UTILIZATION ---
        df['ed_utilization'] = df['num_prior_ed_visits_12m'] + 2 * df['num_prior_admissions_12m']
        df['high_ed_user']   = (df['num_prior_ed_visits_12m'] >= 4).astype(np.int8)
        df['polypharmacy']   = (df['num_active_medications'] >= 5).astype(np.int8)
        
        # --- TEMPORAL FEATURES ---
        df['is_weekend']    = df['arrival_day'].isin(['Saturday', 'Sunday']).astype(np.int8)
        df['is_night']      = df['shift'].eq('night').astype(np.int8)
        df['is_peak_hours'] = df['arrival_hour'].between(10, 18).astype(np.int8)
        
        hour_rad = 2 * np.pi * df['arrival_hour'] / 24
        df['hour_sin'] = np.sin(hour_rad)
        df['hour_cos'] = np.cos(hour_rad)
        
        month_rad = 2 * np.pi * df['arrival_month'] / 12
        df['month_sin'] = np.sin(month_rad)
        df['month_cos'] = np.cos(month_rad)
        
        return df
    
    train = engineer_features(train)
    test  = engineer_features(test)
    
    # --- CRITICAL CHIEF COMPLAINT KEYWORD FLAGS ---
    # Clinically high-risk presentations that should trigger higher acuity
    
    CRITICAL_KEYWORDS = {
        'kw_chest_pain':     r'chest\s*pain|angina|acs|acute coronary',
        'kw_sob':            r'shortness.*breath|dyspn[eo]|sob |respiratory\s*distress|breathing\s*difficult',
        'kw_stroke':         r'stroke|hemipar|aphasia|facial\s*droop|tia|cerebrovascular',
        'kw_seizure':        r'seizure|convuls|epilep|status\s*epilepticus',
        'kw_cardiac_arrest': r'cardiac\s*arrest|asystole|pulseless|cpr|resuscitat',
        'kw_trauma_major':   r'major\s*trauma|polytrauma|mva|motor\s*vehicle|fall\s*from\s*height|penetrat',
        'kw_sepsis':         r'sepsis|septic|bacteremia|bacteria|fungemia',
        'kw_anaphylaxis':    r'anaphyla|massive|severe\s*allerg',
        'kw_suicidal':       r'suicid|self-harm|harm|overdose|intentional|hanging',
        'kw_altered_mental': r'altered\s*mental|confusion|unresponsive|unconscious|syncope|collaps',
        'kw_gi_bleed':       r'haematemes|melena|gi\s*bleed|rectal\s*bleed|blood.*stool|vomit.*blood',
        'kw_fracture':       r'fracture|broken\s*bone|disloc',
        'kw_abdominal':      r'abdominal\s*pain|acute\s*abdomen|appendic',
        'kw_headache':       r'headache|migraine|thunderclap',
        'kw_fever':          r'fever|febrile|pyrexia|rigors',
        'kw_mild':           r'advice|follow-up|prescription|refill|check.?up|minor|mild|chronic\s*stable',
    }

    #Keyword prevalence in training data:
    #kw_chest_pain            : 1,026 patients, mean acuity 2.82
    #kw_sob                   : 1,379 patients, mean acuity 3.07
    #kw_stroke                :   647 patients, mean acuity 2.59
    #kw_seizure               :   180 patients, mean acuity 1.82
    #kw_cardiac_arrest        :    83 patients, mean acuity 1.00
    #kw_trauma_major          :   555 patients, mean acuity 1.73
    #kw_sepsis                : 1,047 patients, mean acuity 2.30
    #kw_anaphylaxis           :   297 patients, mean acuity 1.65
    #kw_suicidal              : 1,278 patients, mean acuity 2.22
    #kw_altered_mental        : 1,769 patients, mean acuity 2.60
    #kw_gi_bleed              :   492 patients, mean acuity 3.19
    #kw_fracture              : 2,063 patients, mean acuity 2.54
    #kw_abdominal             : 1,083 patients, mean acuity 2.80
    #kw_headache              : 1,525 patients, mean acuity 3.63
    #kw_fever                 : 13,067 patients, mean acuity 3.33
    #kw_mild                  : 12,887 patients, mean acuity 4.45
    
    for col_name, pattern in CRITICAL_KEYWORDS.items():
        train[col_name] = train['chief_complaint_raw'].fillna('').str.contains(pattern, case=False, regex=True).astype(np.int8)
        test[col_name]  = test['chief_complaint_raw'].fillna('').str.contains(pattern, case=False, regex=True).astype(np.int8)
    
    kw_cols = [c for c in train.columns if c.startswith('kw_')]
    train['num_critical_keywords'] = train[kw_cols].sum(axis=1)
    test['num_critical_keywords']  = test[kw_cols].sum(axis=1)
    
    # Chief complaint text length and word count
    train['cc_length']     = train['chief_complaint_raw'].fillna('').str.len()
    train['cc_word_count'] = train['chief_complaint_raw'].fillna('').str.split().str.len()
    test['cc_length']      = test['chief_complaint_raw'].fillna('').str.len()
    test['cc_word_count']  = test['chief_complaint_raw'].fillna('').str.split().str.len()


    # --- TF-IDF on chief complaint text ---
    # Dual-channel NLP: word n-grams capture capture local context and structural phrasing (not semantic meaning)
    # character n-grams capture morphological patterns and misspellings
    cc_train = train['chief_complaint_raw'].fillna('unknown')
    cc_test  = test['chief_complaint_raw'].fillna('unknown')
    
    tfidf_word = TfidfVectorizer(
        max_features=max_words, ngram_range=(1, lim_words), analyzer='word', #original (1,3)
        stop_words='english', lowercase=True,
        min_df=3, max_df=0.95, sublinear_tf=True
    )
    tfidf_char = TfidfVectorizer(
        max_features=max_chars, ngram_range=(2, lim_chars), analyzer='char_wb',
        lowercase=True, min_df=5, max_df=0.95, sublinear_tf=True
    )
    
    train_word_mat = tfidf_word.fit_transform(cc_train)
    test_word_mat  = tfidf_word.transform(cc_test)
    train_char_mat = tfidf_char.fit_transform(cc_train)
    test_char_mat  = tfidf_char.transform(cc_test)

    def sanitize_name(s):
        return re.sub(r'[^A-Za-z0-9_]', '_', s).strip('_')
    
    n_word = len(tfidf_word.get_feature_names_out())
    n_char = len(tfidf_char.get_feature_names_out())
    word_raw = [f'tfidf_w_{sanitize_name(n)}' for n in tfidf_word.get_feature_names_out()]
    char_raw = [f'tfidf_c_{sanitize_name(n)}' for n in tfidf_char.get_feature_names_out()]
    all_raw = word_raw + char_raw
    seen = {}
    tfidf_names = []
    for name in all_raw:
        if name in seen:
            seen[name] += 1
            tfidf_names.append(f'{name}_{seen[name]}')
        else:
            seen[name] = 0
            tfidf_names.append(name)
    
    from scipy.sparse import hstack as sp_hstack
    train_tfidf_mat = sp_hstack([train_word_mat, train_char_mat])
    test_tfidf_mat  = sp_hstack([test_word_mat, test_char_mat])
    
    train_tfidf_df = pd.DataFrame(train_tfidf_mat.toarray(), columns=tfidf_names, index=train.index)
    test_tfidf_df  = pd.DataFrame(test_tfidf_mat.toarray(), columns=tfidf_names, index=test.index)
    
    # --- TARGET ENCODING for nurse_id and site_id ---
    # Out-of-fold to prevent leakage
    def target_encode_oof(train_df, test_df, col, target, n_folds=5, seed=42):
        #Target encoding with out-of-fold strategy to prevent leakage.
        global_mean = train_df[target].mean()
        train_enc = pd.Series(np.full(len(train_df), global_mean), index=train_df.index, name=f'{col}_te')
        
        skf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for tr_idx, val_idx in skf.split(train_df, train_df[target]):
            means = train_df.iloc[tr_idx].groupby(col)[target].mean()
            train_enc.iloc[val_idx] = train_df.iloc[val_idx][col].map(means).fillna(global_mean).values
        
        full_means = train_df.groupby(col)[target].mean()
        test_enc = test_df[col].map(full_means).fillna(global_mean)
        test_enc.name = f'{col}_te'
        
        return train_enc, test_enc
    
    for col in ["clean_condition", 'triage_nurse_id', 'site_id']:
        tr_enc, te_enc = target_encode_oof(train, test, col, TARGET)
        train[f'{col}_te'] = tr_enc.values
        test[f'{col}_te']  = te_enc.values

    del tr_enc, te_enc
    gc.collect()
    
    #Range of target encoded of "clean_condition", 'triage_nurse_id', 'site_id'
    #clean_condition_te: train range [1.000, 5.000]
    #triage_nurse_id_te: train range [3.259, 3.391]
    #site_id_te: train range [3.307, 3.335]
    #clean_condition_te ranges widely
    #triage_nurse_id_te and site_id_te have small range
    #This shows site and nurse differences will not play a much role compared to clean_conditio in term of predicting.
    #Nevertheless, we still use triage_nurse_id_te and site_id_te in features for data diversity.
    
    # --- ASSEMBLE FINAL FEATURE MATRIX ---    
    CAT_COLS = [
        'arrival_mode', 'arrival_day', 'arrival_season', 
        'shift',
        'age_group', 'sex', 'language', 'insurance_type',
        'transport_origin', 'pain_location',
        'mental_status_triage',
        'chief_complaint_system',
    ]
    
    NUM_COLS = [
        'arrival_hour', 'arrival_month', 'age',
        'num_prior_ed_visits_12m', 'num_prior_admissions_12m',
        'num_active_medications', 'num_comorbidities',
        'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
        'pulse_pressure', 'heart_rate', 'respiratory_rate',
        'temperature_c', 'spo2', 'gcs_total', 'pain_score',
        'weight_kg', 'height_cm', 'bmi', 'shock_index', 
        'news2_score', "complaint_severity",
    ]
    
    HX_COLS = [c for c in train.columns if c.startswith('hx_')]
    
    ENG_COLS = (
        [c for c in train.columns if c.startswith('flag_')]
        + ['num_abnormal_vitals', 'qsofa_score', 'sirs_count',
           'comorbidity_burden', 'cv_risk_score', 'respiratory_risk',
           'metabolic_risk', 'psych_risk',
           'age_x_shock_index', 'age_x_news2', 'age_x_gcs',
           'age_x_comorbidities', 'age_x_qsofa',
           'hr_rr_ratio', 'sbp_hr_product', 'map_hr_ratio',
           'pp_sbp_ratio', 'spo2_rr_product',
           'ed_utilization', 'high_ed_user', 'polypharmacy',
           'is_weekend', 'is_night', 'is_peak_hours',
           'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
           'triage_nurse_id_te', 'site_id_te',
           'num_critical_keywords', 'cc_length', 'cc_word_count']
        + kw_cols
    )
    ENG_COLS = list(dict.fromkeys(ENG_COLS))  # deduplicate
    
    # Encode categoricals
    for col in CAT_COLS:
        combined = pd.concat([train[col], test[col]], axis=0).astype('category')
        codes = combined.cat.codes
        train[col] = codes.iloc[:len(train)].values
        test[col]  = codes.iloc[len(train):].values

    CC_COLS = ["clean_condition_te"]

    all_feature_cols = CAT_COLS + NUM_COLS + HX_COLS + ENG_COLS + CC_COLS
    all_feature_cols = [c for c in all_feature_cols if c in train.columns]
    
    X_train = pd.concat([train[all_feature_cols].reset_index(drop=True),
                        train_tfidf_df.reset_index(drop=True)],axis=1)
    X_test = pd.concat([test[all_feature_cols].reset_index(drop=True),
                       test_tfidf_df.reset_index(drop=True)],axis=1)
    
    y_train = train[TARGET].values
    
    from sklearn.preprocessing import OneHotEncoder
    
    all_data = pd.concat([X_train, X_test], axis=0)

    
    # Make category from numeric features
    # When unique values is less than 10 add category features from it
    add_cat_cols = []
    for col in NUM_COLS:
        how_many = len(all_data[col].unique())
        if how_many < 10:
            X_train[col + "_trans2cat"] = X_train[col].copy().fillna("-1").astype("str")
            X_test[col + "_trans2cat"] = X_test[col].copy().fillna("-1").astype("str")
            add_cat_cols.append(col + "_trans2cat")
    for col in add_cat_cols:
        X_train[col] = X_train[col].astype("str")
        X_test[col] = X_test[col].astype("str")
    
    
    # one-hot encoder
    one_hot_encoder = OneHotEncoder(sparse_output=False)
    one_hot_features = CAT_COLS + add_cat_cols
    one_hot_encoder.fit(X_train[one_hot_features])
    
    features_new_train = one_hot_encoder.transform(X_train[one_hot_features])
    features_new_train = pd.DataFrame(features_new_train)
    features_new_train.columns = ['onehot_'+ str(x) for x in range(len(features_new_train.columns))]
    X_train = pd.concat([X_train, features_new_train], axis=1)
    
    features_new_test = one_hot_encoder.transform(X_test[one_hot_features])
    features_new_test = pd.DataFrame(features_new_test)
    features_new_test.columns = ['onehot_'+ str(x) for x in range(len(features_new_test.columns))]
    X_test = pd.concat([X_test, features_new_test], axis=1)
    
    
    # Encode categoricals of add_cat_cols
    for col in add_cat_cols:
        combined = pd.concat([X_train[col], X_test[col]], axis=0).astype('category')
        codes = combined.cat.codes
        X_train[col] = codes.iloc[:len(train)].values
        X_test[col]  = codes.iloc[len(train):].values
    
    
    # Recheck there is no Leakage.
    # Make sure disposition and ed_los_hours are not included in both train and test data.
    # Because disposition and ed_los_hours can only be obtained after ESI level is predicted.
    train_features = list(X_train.columns)
    test_features = list(X_test.columns)
    
    if "disposition" in train_features:
        print(f"Leakage")
    if "ed_los_hours" in train_features:
        print(f"Leakage")
    if "triage_aquity" in train_features:
        print(f"Leakage")
    if "disposition" in test_features:
        print(f"Leakage")
    if "ed_los_hours" in test_features:
        print(f"Leakage")
    if "triage_aquity" in test_features:
        print(f"Leakage")
    
    ONE_HOT_ENCODER_COLS = list(features_new_train.columns)

    
    # SentenceTransformer embedding (NLP) 
    from sentence_transformers import SentenceTransformer
    
    TRAIN_PATH  = "/kaggle/input/competitions/triagegeist/train.csv"
    TEST_PATH   = "/kaggle/input/competitions/triagegeist/test.csv"
    CC_PATH     = "/kaggle/input/competitions/triagegeist/chief_complaints.csv"
    SUB_PATH    = "/kaggle/input/competitions/triagegeist/sample_submission.csv"
    
    train_df = pd.read_csv(TRAIN_PATH)
    test_df  = pd.read_csv(TEST_PATH)
    cc_df    = pd.read_csv(CC_PATH)
    
    train_df = train_df.merge(cc_df, on="patient_id", how="left")
    test_df  = test_df.merge(cc_df,  on="patient_id", how="left")

    # Fill missing complaints
    train_df["chief_complaint_raw"].fillna("unknown complaint", inplace=True)
    test_df["chief_complaint_raw"].fillna("unknown complaint",  inplace=True)
    
    def clean_complaint(text: str) -> str:
        #Light clinical text cleaning:
        #- Lowercase
        #- Remove punctuation / digits (keep letters & spaces)
        #- Strip extra whitespace
        #- Preserve key clinical abbreviations by expanding them first
        
        ABBREV = {
            r"\bdob\b": "difficulty of breathing",
            r"\bloc\b": "loss of consciousness",
            r"\bn/v\b": "nausea vomiting",
            r"\bhtn\b": "hypertension",
            r"\bdm\b":  "diabetes",
            r"\beti\b": "endotracheal intubation",
            r"\brta\b": "road traffic accident",
            r"\bmva\b": "motor vehicle accident",
        }
        text = str(text).lower()
        for pattern, expansion in ABBREV.items():
            text = re.sub(pattern, expansion, text)
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    
    train_df["complaint_clean"] = train_df["chief_complaint_raw"].apply(clean_complaint)
    test_df["complaint_clean"]  = test_df["chief_complaint_raw"].apply(clean_complaint)
    
    EMBED_MODEL = "all-MiniLM-L6-v2"   # fast & strong general-purpose model
    #Loading SentenceTransformer
    st_model = SentenceTransformer(EMBED_MODEL)
    
    def encode_texts(texts, batch_size=256, show_progress=True):
        return st_model.encode(
            list(texts),
            batch_size=batch_size,
            show_progress_bar=show_progress,
            normalize_embeddings=True   # cosine-compatible
        )
    
    #Encoding training complaints
    X_emb_train = encode_texts(train_df["complaint_clean"].values)
    #Encoding test complaints
    X_emb_test  = encode_texts(test_df["complaint_clean"].values) 
    
    # Convert to DataFrame
    X_emb_train = pd.DataFrame(X_emb_train)
    X_emb_train.columns = ["emb_" + str(x) for x in range(len(X_emb_train.columns))]
    X_emb_test = pd.DataFrame(X_emb_test)
    X_emb_test.columns = ["emb_" + str(x) for x in range(len(X_emb_test.columns))]

    EMB_TRAIN_COLS = list(X_emb_train.columns)
    EMB_TEST_COLS = list(X_emb_test.columns)
    
    # Concat
    X_train = pd.concat([X_train, X_emb_train], axis=1)
    X_test = pd.concat([X_test, X_emb_test], axis=1)
    
    del train_df, test_df, cc_df
    gc.collect()
    
    
    #Scaling for nn models actually but gbdt can use it   
    X_train_filled = X_train.fillna(0)
    X_test_filled = X_test.fillna(0)
    scaler = StandardScaler()
    
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_filled), columns=X_train.columns, index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_filled), columns=X_test.columns, index=X_test.index
    )
    
    #Convert from 64-bit to 32-bit data type to save memory
    X_train_scaled = X_train_scaled.astype("float32")
    X_test_scaled = X_test_scaled.astype("float32")
    y_train = y_train.astype(np.float32)
    
    cols_dict = {"CAT_COLS":CAT_COLS, 
                 "add_cat_cols":add_cat_cols, 
                 "NUM_COLS":NUM_COLS, 
                 "HX_COLS":HX_COLS,
                 "ENG_COLS":ENG_COLS, 
                 "CC_COLS":CC_COLS,
                 "ONE_HOT_ENCODER_COLS":ONE_HOT_ENCODER_COLS, 
                 "EMB_TRAIN_COLS":EMB_TRAIN_COLS,
                 "EMB_TEST_COLS":EMB_TEST_COLS,
                 "tfidf_names":tfidf_names}

    return X_train_scaled, X_test_scaled, y_train, train, test, cols_dict


CPU times: user 15 µs, sys: 0 ns, total: 15 µs
Wall time: 17.6 µs


In [6]:
%%time


X_train_scaled, X_test_scaled, y_train, _, _, cols_dict = features_function(max_words=50,max_chars=40,lim_words=3,lim_chars=3)
X_train_scaled = X_train_scaled[cols_dict["CAT_COLS"] + cols_dict["NUM_COLS"] + cols_dict["CC_COLS"] + cols_dict["tfidf_names"]]
X_test_scaled = X_test_scaled[cols_dict["CAT_COLS"] + cols_dict["NUM_COLS"] + cols_dict["CC_COLS"] + cols_dict["tfidf_names"]]

print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]


CPU times: user 41.9 s, sys: 3.17 s, total: 45.1 s
Wall time: 43.6 s


In [7]:
print(X_train_scaled.shape)
display(X_train_scaled.head(2))

(80000, 126)


,arrival_mode,arrival_day,arrival_season,shift,age_group,sex,language,insurance_type,transport_origin,pain_location,...,tfidf_c_st,tfidf_c_t,tfidf_c_te,tfidf_c_th,tfidf_c_th_1,tfidf_c_ti,tfidf_c_ve,tfidf_c_wi_1,tfidf_c_wit,tfidf_c_y
0,0.884776,-0.997779,-0.445836,0.3672,-0.347674,0.879949,-0.128107,0.590565,0.493733,-0.384289,...,-0.727,0.741196,-0.806376,1.776078,0.849577,-0.917574,1.201051,0.833682,0.833682,-0.648762
1,0.884776,0.502744,-0.445836,0.3672,-1.199084,-0.952658,1.065393,-2.975649,-1.506629,-0.384289,...,-0.727,0.771841,1.764308,-1.150207,-0.973432,0.645452,-0.669432,-0.982266,-0.982266,-0.648762


In [8]:
print(X_test_scaled.shape)
display(X_test_scaled.head(2))

(20000, 126)


,arrival_mode,arrival_day,arrival_season,shift,age_group,sex,language,insurance_type,transport_origin,pain_location,...,tfidf_c_st,tfidf_c_t,tfidf_c_te,tfidf_c_th,tfidf_c_th_1,tfidf_c_ti,tfidf_c_ve,tfidf_c_wi_1,tfidf_c_wit,tfidf_c_y
0,0.884776,-0.997779,-0.445836,0.367200,1.355148,0.879949,-1.321608,0.590565,0.493733,0.776544,...,1.085413,-0.858466,-0.806376,0.688835,-0.973432,-0.917574,-0.669432,-0.982266,-0.982266,1.224586
1,-1.385655,-0.997779,-0.445836,-1.423854,-0.347674,-0.952658,1.065393,-0.598173,-1.006539,1.163489,...,0.861317,-0.858466,0.582816,0.461446,0.726528,2.083709,-0.669432,0.711109,0.711109,0.992956


In [9]:
print(len(y_train))
y_train[:3]

80000


array([7.35, 0.7 , 0.63], dtype=float32)

## RealMLP Regressor

In [10]:
%%time


import random 
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

def realmlp_function(num):
    seed_everything(seed=num)
    #print("✅ Configuration complete!")
    
    #print(f'Training RealMLP Regressor {N_FOLDS}-fold stratified CV')
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=num)
    oof_realmlp_reg = np.zeros(len(X_train_scaled))
    test_realmlp_reg = np.zeros(len(X_test_scaled))
    
    realmlp_params = {
        'device': 'cuda',
        'random_state': num,
        'verbosity': 0,
        'n_epochs': 5,  
        'batch_size': 256,
        'n_ens': 8,
        'val_metric_name': 'mae',
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 6,
        'first_layer_lr_factor': 0.25,
        'hidden_sizes': "rectangular",
        'hidden_width': 352,
        'lr': 0.075,
        'ls_eps': 0.01,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 4,
        'p_drop': 0.05,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151,
        'plr_sigma': 2.33,
        'scale_lr_factor': 2.24,
        'sq_mom': 0.988,
        'wd': 0.0236,
    }
    
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_scaled, y_train)):    
        
        X_tr = X_train_scaled.iloc[tr_idx]
        X_val = X_train_scaled.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
        model = RealMLP_TD_Regressor(**realmlp_params)
        model.fit(X_tr, y_tr,
                  X_val=X_val, y_val=y_val,
                 )
    
        oof_realmlp_reg[val_idx] = model.predict(X_val)
        test_realmlp_reg += model.predict(X_test_scaled) / N_FOLDS
    
        # save model
        model_info = {'train_index':tr_idx,'eval_index':val_idx,'model':model}
        with open(model_dir+str(num)+'_realmlp_reg_fold%s.pkl'%(fold),'wb') as f:
            pickle.dump(model_info,f)
                
        val_pred = oof_realmlp_reg[val_idx]
        mae = mean_absolute_error(y_val, val_pred)
        #print(f'  Fold {fold+1}: MAE={mae:.6f} ')
        
        del model
        gc.collect()
    
    realmlp_oof_regs = oof_realmlp_reg
    realmlp_oof_mae = mean_absolute_error(y_train, realmlp_oof_regs)
    print(f'\nRealMLP Reg OOF: MAE={realmlp_oof_mae:.6f} (seed={num})')
    
    return realmlp_oof_regs, realmlp_oof_mae, test_realmlp_reg, num

realmlp_oofs = []
realmlp_maes = []
realmlp_tests = []
seeds = []

my_list = []
for i in range(125):
    oof, mae, test, seed_num = realmlp_function(i)
    my_list.append((oof, mae, test, seed_num))

sorted1 = sorted(my_list, key=lambda item: item[1], reverse=False)
top_chosen = 5
sorted1 = sorted1[:top_chosen]
realmlp_oofs  = [item[0] for item in sorted1]
realmlp_maes  = [item[1] for item in sorted1]
realmlp_tests = [item[2] for item in sorted1]
seeds         = [item[3] for item in sorted1]

#stats
realmlp_maes_avg = np.mean(realmlp_maes)
realmlp_maes_std = np.std(realmlp_maes)
realmlp_maes_min = np.min(realmlp_maes)
realmlp_maes_max = np.max(realmlp_maes)
print(f'Avg RealMLP oof MAE = {realmlp_maes_avg:.6f}')
print(f'Std RealMLP oof MAE = {realmlp_maes_std:.6f}')
print(f'Min RealMLP oof MAE = {realmlp_maes_min:.6f}')
print(f'Max RealMLP oof MAE = {realmlp_maes_max:.6f}')

realmlp_results_info = {
    "realmlp_oofs":realmlp_oofs,
    "realmlp_maes":realmlp_maes,
    "realmlp_tests":realmlp_oofs,
    "seeds":seeds
}

with open("realmlp_results_info.pkl","wb") as f:
    pickle.dump(realmlp_results_info,f)
    
print()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics an


RealMLP Reg OOF: MAE=1.145009 (seed=0)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics an


RealMLP Reg OOF: MAE=1.144519 (seed=1)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics an

...


RealMLP Reg OOF: MAE=1.144510 (seed=122)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics an


RealMLP Reg OOF: MAE=1.144598 (seed=123)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics an


RealMLP Reg OOF: MAE=1.144869 (seed=124)
Avg RealMLP oof MAE = 1.144241
Std RealMLP oof MAE = 0.000037
Min RealMLP oof MAE = 1.144168
Max RealMLP oof MAE = 1.144266

CPU times: user 2h 54min 16s, sys: 9min 14s, total: 3h 3min 31s
Wall time: 3h 2min 49s


In [11]:
del X_train_scaled, X_test_scaled
gc.collect()


print()

## ensemble

In [12]:
%%time
    
def stacking_function(num):    
    oof_meta_reg = np.column_stack((
        *realmlp_oofs,
    ))
    
    test_meta_reg = np.column_stack((
        *realmlp_tests,
    ))
    
    #Boolean Indexing (In-place): efficient for large arrays and offers granular control
    oof_meta_reg[np.isnan(oof_meta_reg)] = 0
    test_meta_reg[np.isnan(test_meta_reg)] = 0

    kf = KFold(n_splits=5, shuffle=True, random_state=num)
    
    # CAT STACK
    oof_cat_stacked_reg = np.zeros(len(oof_meta_reg))
    test_cat_stacked_reg = np.zeros(len(test_meta_reg))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(oof_meta_reg, y_train)):
        cat_stack = CatBoostRegressor(
            iterations=250,
            depth=7,
            loss_function="RMSE",
            random_seed=num,
            task_type="GPU",
            verbose=0,
            early_stopping_rounds=10,
        )
        cat_stack.fit(oof_meta_reg[tr_idx], y_train[tr_idx])
    
        # save model
        model_info = {'train_index':tr_idx,'eval_index':val_idx,'model':cat_stack}
        with open(model_dir+'cat_stack_reg_fold%s.pkl'%(fold),'wb') as f:
            pickle.dump(model_info,f)
            
        oof_cat_stacked_reg[val_idx] = cat_stack.predict(oof_meta_reg[val_idx])
        test_cat_stacked_reg += cat_stack.predict(test_meta_reg) / N_FOLDS
        fold_pred = oof_cat_stacked_reg[val_idx]
        #print(f'  Fold {fold+1}: CAT stacked MAE={mean_absolute_error(y_train[val_idx], fold_pred):.6f}  ')
    
    cat_stacked_reg = oof_cat_stacked_reg
    cat_stacked_mae = mean_absolute_error(y_train, cat_stacked_reg)
    
    # LinearRegression STACK
    oof_linreg_stacked_reg = np.zeros(len(oof_meta_reg))
    test_linreg_stacked_reg = np.zeros(len(test_meta_reg))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(oof_meta_reg, y_train)):
        linreg_stack = LinearRegression()
        linreg_stack.fit(oof_meta_reg[tr_idx], y_train[tr_idx])
    
        # save model
        model_info = {
            'train_index': tr_idx,
            'eval_index': val_idx,
            'model': linreg_stack}
        with open(model_dir+'linreg_stack_reg_fold%s.pkl'%(fold),'wb') as f:
            pickle.dump(model_info,f)
            
        oof_linreg_stacked_reg[val_idx] = linreg_stack.predict(oof_meta_reg[val_idx])
        test_linreg_stacked_reg += cat_stack.predict(test_meta_reg) / N_FOLDS
        fold_pred = oof_linreg_stacked_reg[val_idx]
        #print(f'  Fold {fold+1}: LinReg stacked MAE={mean_absolute_error(y_train[val_idx], fold_pred):.6f}  ')
    
    linreg_stacked_reg = oof_linreg_stacked_reg
    linreg_stacked_mae = mean_absolute_error(y_train, linreg_stacked_reg)
    
    # Final ensemble
    w1 = 0.0
    w2 = 1.0
    stacked_reg = w1 * cat_stacked_reg + w2 * linreg_stacked_reg
    stacked_mae = mean_absolute_error(y_train, stacked_reg)
    test_ensemble_reg = w1 * test_cat_stacked_reg + w2 * test_linreg_stacked_reg

    print()
    print("-"*60)
    print(f'  Average RealMLP oof:       \t\tMAE={realmlp_maes_avg:.6f}')
    print("-"*60)
    print(f'  Cat stacked:             \t\tMAE={cat_stacked_mae:.6f}')
    print(f'  LinReg stacked:          \t\tMAE={linreg_stacked_mae:.6f}')
    print("-"*60)
    print(f'  Hybrid stacked:          \t\tMAE={stacked_mae:.6f}')
    print("-"*60)
    return stacked_mae, test_ensemble_reg

_, test_ensemble_reg = stacking_function(42)


------------------------------------------------------------
  Average RealMLP oof:       		MAE=1.144241
------------------------------------------------------------
  Cat stacked:             		MAE=1.151636
  LinReg stacked:          		MAE=1.144194
------------------------------------------------------------
  Hybrid stacked:          		MAE=1.144194
------------------------------------------------------------
CPU times: user 21 s, sys: 5.32 s, total: 26.3 s
Wall time: 9.99 s


In [13]:
%%time

test_raw  = pd.read_csv(f'{DATA}/test.csv')
submission = pd.DataFrame({
    ID: test_raw[ID],
    TARGET: test_ensemble_reg,
})
submission.to_csv('submission.csv', index=False)
submission.head()

CPU times: user 122 ms, sys: 2.94 ms, total: 125 ms
Wall time: 130 ms


,patient_id,ed_los_hours
0,TG-FZUFCRZS3,5.893606
1,TG-SSCOXTYI1,4.016914
2,TG-JY74ZR35D,6.185479
3,TG-JDKD5G62X,2.014272
4,TG-J1BSAAXR0,2.006503


In [14]:
#linearregression MAE=1.148006
#elasticnet l1 ratio 0 MAE=1.241338
#elasticnet l1 ratio 0.5 MAE=1.320144
#elasticnet l1 ratio 1.0 MAE=1.357449
#CatBoost only: MAE=1.146998